# Exercise 16.1: Solving the bistable equation in Python

We want to solve the bistable equation on a 1D cable of length $L = 100$:

$$
\frac{\partial V}{\partial t} = k \frac{\partial^2 V}{\partial x^2} + A V(1 - V)(V - \alpha)
$$

We will use the following parameter values:

- $k = 2.0$ (Diffusion constant)
- $A = 1.0$ (Reaction scaling)
- $\alpha = 0.1$ (Activation threshold)
- $\Delta x = 1$ (Spatial step)
- $\Delta t = 0.1$ ms (Time step)

We will apply an initial stimulus to the far left edge of the cable ($V = 0.3$ for the first 10% of the cable) to trigger the wave. The boundary conditions at the absolute ends of the cable ($x=0$ and $x=L$) will be sealed (no current can flow out the ends).


## Exercise 16.1a: Explicit scheme with loops

Complete the explicit update scheme inside the loop below.

You need to use the `v_prev` array (which holds the voltage from the previous time step $n$) to calculate the spatial diffusion and the reaction term, and then save the new values into the `v` array (which represents step $n+1$).

_Note: The boundary conditions at $j=0$ and $j=N$ have been provided for you to handle the sealed ends._


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython import display
import time

# Parameters
k = 2.0
A = 1.0
alpha = 0.1
L = 100

dx = 1
dt = 0.1
N = int(L / dx)

# Initialize arrays
v = np.zeros(N + 1)
left = int(N / 10)
v[:left] = 0.3  # Apply the initial stimulus to the left side!


def f(V):
    return A * V * (1 - V) * (V - alpha)


# We use v_prev to hold the known values from the previous time step
v_prev = np.copy(v)

# Simulation loop
for i in range(1400):
    # 1. Update the internal nodes using the explicit FDM scheme
    for j in range(1, N):
        # Your code here: calculate I_ion, diffusion, and the new v[j]
        I_ion = ...
        diff = ...
        v[j] = ...

    # 2. Update the boundary nodes (sealed ends)
    v[0] = (
        v_prev[0] + dt * (k / dx**2) * 2 * (v_prev[1] - v_prev[0]) + dt * f(v_prev[0])
    )
    v[N] = (
        v_prev[N]
        + dt * (k / dx**2) * 2 * (v_prev[N - 1] - v_prev[N])
        + dt * f(v_prev[N])
    )

    # 3. Save the newly calculated state for the next loop iteration
    v_prev = np.copy(v)

    # 4. Live Plotting (updates every 20 steps)
    if i % 20 == 0:
        plt.clf()
        plt.axis([0, L, 0, 1.1])
        plt.plot(v, color="C0", linewidth=2)
        plt.title(f"Time step i={i}")
        display.clear_output(wait=True)
        display.display(plt.gcf())
        time.sleep(0.01)

## Exercise 16.1b: Vectorization for speed

Standard `for` loops in Python are notoriously slow. In computational physiology, tracking thousands of grid points using Python loops can bring your simulation to a grinding halt.

Instead, we can use **NumPy vectorization**. By passing entire slices of arrays into our FDM formula at once, NumPy handles the loop internally in highly optimized C-code, speeding up the calculation by orders of magnitude!

Complete the code below to implement the exact same scheme, but without the internal spatial `for` loop. We have set up the slicing indices `I`, `Ip` (I plus 1), and `Im` (I minus 1) for you.


In [ ]:
# Reset initial conditions
v = np.zeros(N + 1)
v[:left] = 0.3

# Introduce the appropriate arrays for slicing the internal nodes
I = np.arange(1, N)  # Current nodes
Ip = I + 1  # Right neighbors
Im = I - 1  # Left neighbors

for i in range(1400):
    # 1. Calculate the reaction term for ALL nodes simultaneously
    I_ion = ...

    # We must explicitly save the old state before doing vectorized in-place updates!
    v_prev = np.copy(v)

    # 2. Add diffusion to the internal nodes using array slicing
    v[I] = v_prev[I] + dt * (k / dx**2) * (...)

    # 3. Update the boundary nodes
    v[0] = v_prev[0] + dt * (k / dx**2) * 2 * (v_prev[1] - v_prev[0])
    v[N] = v_prev[N] + dt * (k / dx**2) * 2 * (v_prev[N - 1] - v_prev[N])

    # 4. Add the reaction term to the entire array at once
    v = v + dt * I_ion

    # Live Plotting
    if i % 20 == 0:
        plt.clf()
        plt.axis([0, L, 0, 1.1])
        plt.plot(v, color="C3", linewidth=2)
        plt.title(f"Time step i={i} (Vectorized)")
        display.clear_output(wait=True)
        display.display(plt.gcf())
        time.sleep(0.01)